# Loading in our files:

In [47]:
import pandas as pd
import numpy as np
print(pd.__version__)
print(np.__version__)

3.0.0
2.4.2


In [48]:
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm


In [49]:
files = sorted(Path("processed_chunks_paquet").glob("chunk_*.parquet"))

for file in files[:3]:
    df = pd.read_parquet(file)
    
    print(file.name)
    print(df.head())

chunk_000.parquet
   Unnamed: 0         id               domain        type  \
0         732  7444726.0   nationalreview.com   political   
1        1348  6213642.0    beforeitsnews.com        fake   
2        7119  3867639.0     dailycurrant.com      satire   
3        1518  9560791.0          nytimes.com    reliable   
4        9345  2059625.0  infiniteunknown.net  conspiracy   

                                                 url  \
0  http://www.nationalreview.com/node/152734/%E2%...   
1  http://beforeitsnews.com/economy/2012/06/the-c...   
2  http://dailycurrant.com/2016/01/18/man-awoken-...   
3  https://query.nytimes.com/gst/fullpage.html?re...   
4  http://www.infiniteunknown.net/2011/09/14/100-...   

                   scraped_at                 inserted_at  \
0  2017-11-27T01:14:42.983556  2018-02-08 19:18:34.468038   
1    2017-11-27T01:14:08.7454  2018-02-08 19:18:34.468038   
2  2017-11-27T01:14:21.395055  2018-02-07 23:39:33.852671   
3  2018-02-11 00:46:42.632962  201

# Loading and splitting data:

In [56]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

fake_categories = {"fake", "hate", "unreliable", "conspiracy", "junksci", "rumor", "bias"}
valid_types = fake_categories | {"reliable", "political", "clickbait"}

def load_chunks_logistic(files):
    dfs = []
    for file in tqdm(files):
        df = pd.read_parquet(file)
        df = df[df["type"].isin(valid_types)][["type", "tokens_stemmed", "domain"]]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data["text"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))
    data["text_and_domain"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens)) + " " + data["domain"]
    data["label"] = data["type"].apply(lambda x: 1 if x in fake_categories else 0)
    return data[["text", "text_and_domain", "label"]]

files = sorted(Path("processed_chunks_paquet").glob("chunk_*.parquet"))
all_data = load_chunks_logistic(files[:100])

# First split off 20% for val+test
train_data, temp_data = train_test_split(all_data, test_size=0.2, random_state=42)

# Split the 20% evenly into val and test
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

X_train, y_train = train_data["text"], train_data["label"]
X_val,   y_val   = val_data["text"],   val_data["label"]
X_test,  y_test  = test_data["text"],  test_data["label"]

X_tr_domain = train_data["text_and_domain"]
X_v_domain  = val_data["text_and_domain"]
X_te_domain = test_data["text_and_domain"]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")



  0%|          | 0/100 [00:00<?, ?it/s]

Train: 712000, Val: 89000, Test: 89001


# Simple logistic regression

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer #Doesnt do any normalization

# Vectorize
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

# sm = SMOTE()  
# X_train_re, y_train_re = sm.fit_resample(X_train_vec, y_train)

# Train
model = LogisticRegression(
  random_state = 42,
  C=1,
  max_iter=1000,
)
model.fit(X_train_vec, y_train)

# Evaluate on validation set
y_val_pred = model.predict(X_val_vec)
print("Validation:")
print(classification_report(y_val, y_val_pred, target_names=["reliable", "fake"]))

Validation:
              precision    recall  f1-score   support

    reliable       0.87      0.85      0.86     44108
        fake       0.85      0.88      0.87     44892

    accuracy                           0.86     89000
   macro avg       0.86      0.86      0.86     89000
weighted avg       0.86      0.86      0.86     89000



In [58]:
y_test_pred = model.predict(X_test_vec)
print("Validation:")
print(classification_report(y_test, y_test_pred, target_names=["reliable", "fake"]))

Validation:
              precision    recall  f1-score   support

    reliable       0.87      0.85      0.86     43948
        fake       0.85      0.88      0.87     45053

    accuracy                           0.86     89001
   macro avg       0.86      0.86      0.86     89001
weighted avg       0.86      0.86      0.86     89001



In [59]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer #Doesnt do any normalization

# Vectorize
vectorizer = CountVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_tr_domain)
X_val_vec   = vectorizer.transform(X_v_domain)
X_test_vec  = vectorizer.transform(X_te_domain)

# Train
model_two = LogisticRegression(max_iter=1000)
model_two.fit(X_train_vec, y_train)

# Evaluate on validation set
y_val_pred_two = model_two.predict(X_val_vec)
print("Validation:")
print(classification_report(y_val, y_val_pred_two, target_names=["reliable", "fake"]))

#Lasse's idé.
avg_tokens = train_data["text"].apply(lambda x: len(x.split())).mean()
print(f"Average token length: {avg_tokens:.1f}")

Validation:
              precision    recall  f1-score   support

    reliable       0.98      0.97      0.98     44108
        fake       0.97      0.98      0.98     44892

    accuracy                           0.98     89000
   macro avg       0.98      0.98      0.98     89000
weighted avg       0.98      0.98      0.98     89000

Average token length: 251.9


In [60]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd
from sklearn.metrics import classification_report

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize_text(text):
    text = str(text).lower()
    return word_tokenize(text)

def clean_tokens(tokens):
    return [t for t in tokens if t.isalpha() and t not in stop_words]

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

df_liar = pd.read_csv("liar_dataset/test.tsv", sep="\t", header=None)
df_liar.columns = ["id", "label", "statement", "subject", "speaker", 
                   "job", "state", "party", "barely_true", "false_count",
                   "half_true", "mostly_false", "pants_fire", "context"]

fake_categories_liar = {"false", "pants-on-fire", "barely-true"}
valid_types_liar = fake_categories_liar | {"true", "mostly-true"}

def map_to_binary(label):
    if label in fake_categories_liar:
        return 1
    elif label in valid_types_liar:
        return 0
    else:
        return None

df_liar["binary_label"] = df_liar["label"].apply(map_to_binary)
df_liar = df_liar.dropna(subset=["binary_label"])
df_liar["binary_label"] = df_liar["binary_label"].astype(int)

TEXT_COL = "statement"

df_liar[TEXT_COL] = df_liar[TEXT_COL].fillna("")

# Preprocess LIAR text
df_liar["tokens"] = df_liar[TEXT_COL].apply(tokenize_text)
df_liar["tokens_clean"] = df_liar["tokens"].apply(clean_tokens)
df_liar["tokens_stemmed"] = df_liar["tokens_clean"].apply(stem_tokens)
df_liar["text"] = df_liar["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))

# Use the preprocessed text here
liar_vec = vectorizer.transform(df_liar["text"])
y_liar = df_liar["binary_label"]

print(classification_report(y_liar, model.predict(liar_vec)))

#Lasse's idé. LIAR
avg_tokens = df_liar["text"].apply(lambda x: len(x.split())).mean()
print(f"Average token length: {avg_tokens:.1f}")

              precision    recall  f1-score   support

           0       0.46      0.42      0.44       449
           1       0.48      0.52      0.50       461

    accuracy                           0.47       910
   macro avg       0.47      0.47      0.47       910
weighted avg       0.47      0.47      0.47       910

Average token length: 10.1


In [61]:
print(df_liar[["statement", "text"]].head(1))

                                           statement  \
0  Building a wall on the U.S.-Mexico border will...   

                                text  
0  build wall border take liter year  
